# Extracting Linguistic Features using `spaCy`

Feedback/questions: pushpak.karnick@digipen.edu

This notebook will cover the extraction and processing of some os `spaCy`'s common linguistic features. They include:
- part-of-speech (POS) tagger
- the dependency parser
- named entity recognizer (NER)
- merging/splitting of features



## Part-of-Speech (POS) Tagging

A part-of-speech (POS) is a category or role played by individual words, tokens, or phrases, in a sentence. A POS defines the syntactic as well as semantic relationships between tokens. The English language has 9 main categories for POS:

- Verb: Expresses an action or a state of being
- Noun: Identifies a person, a place, or a thing, or names a particular of one of these (a proper noun)
- Pronoun: Can replace a noun or noun phrase
- Determiner: Is placed in front of a noun to express a quantity or clarify what the noun refers to—briefly, a noun introducer
- Adjective: Modifies a noun or a pronoun
- Adverb: Modifies a verb, an adjective, or another adverb
- Preposition: Connects a noun/pronoun to other parts of the sentence
- Conjunction: Glues words, clauses, and sentences together
- Interjection: Expresses emotion in a sudden and exclamatory way

Each category, and its derivations according to valid English grammar rules, is expressed as a particular tag within spaCy

In [2]:
import spacy

# Load the spacy English model
nlp = spacy.load('en_core_web_sm')

inputText = """Mr. Sherlock Holmes, who was usually very late in the mornings,
      save upon those not infrequent occasions when he was up all
      night, was seated at the breakfast table. I stood upon the
      hearth - rug and picked up the stick which our visitor had left
      behind him the night before."""

doc = nlp(inputText)

for token in doc:
    print("\t\t".join([token.text, token.pos_, spacy.explain(token.pos_)]))

Mr.		PROPN		proper noun
Sherlock		PROPN		proper noun
Holmes		PROPN		proper noun
,		PUNCT		punctuation
who		PRON		pronoun
was		AUX		auxiliary
usually		ADV		adverb
very		ADV		adverb
late		ADJ		adjective
in		ADP		adposition
the		DET		determiner
mornings		NOUN		noun
,		PUNCT		punctuation

      		SPACE		space
save		VERB		verb
upon		SCONJ		subordinating conjunction
those		DET		determiner
not		PART		particle
infrequent		ADJ		adjective
occasions		NOUN		noun
when		SCONJ		subordinating conjunction
he		PRON		pronoun
was		AUX		auxiliary
up		ADV		adverb
all		DET		determiner

      		SPACE		space
night		NOUN		noun
,		PUNCT		punctuation
was		AUX		auxiliary
seated		VERB		verb
at		ADP		adposition
the		DET		determiner
breakfast		NOUN		noun
table		NOUN		noun
.		PUNCT		punctuation
I		PRON		pronoun
stood		VERB		verb
upon		SCONJ		subordinating conjunction
the		DET		determiner

      		SPACE		space
hearth		ADJ		adjective
-		PUNCT		punctuation
rug		NOUN		noun
and		CCONJ		coordinating conjunction
picked		VERB

POS tagging helps identify the nouns (actors), verbs (actions), adjectives (descriptors) and other meaningful components of the language in order to record semantic information ('facts') from the text.

They also serve a vital role in resolving ambiguity when a word may be used as more than one POS.

POS tagging has a rich history in the field of Natural Language Understanding (NLU). SpaCy implements a statistical model to determine the POS of each token in a sentence. The inferred POS tag for a token is dependent on the preceding context - the preceding words, their tags, and the text of the word itself. Methods employed to train the model include:
- Sequence-to-Sequence Learning
- Hidden Markov Models
- Neural Networks using Long Short-Term Memory (LSTM) cells


### Word-Sense Disambiguation (WSD)

WSD is a classical problem in NLU that aims to assign a sense (meaning) to a word based on its context. The complexity of modern languages allows one word to employed in a variety of contexts. E.g. the word 'bass' can refer to a musical instrument, a type of fish, a vocalist, or a specific vocal range. Determining which meaning to ascribe to a specific usage of the word is usually dependent on the surrounding context.

Let us see if a POS tagger is able help us resolve the ambiguity for the word 'bass':


In [3]:
def getPOSInformation(word, inputSentences):
    doc = []
    for sent in inputSentences:
        doc.append( nlp(sent) )

    for sentence in doc:
        for token in sentence:
            if token.lower_ == word or word == "*": # Specific word or all words
                print(f"Token: {token.text} - POS: {token.pos_}, {spacy.explain(token.tag_)} - DEP: {token.dep_}")

In [4]:
inputText = [ 'I caught a bass in the lake',
                   'I can sing bass',
                   'I can play bass',
                   'I shop at Bass Pro']

getPOSInformation('bass', inputText)

Token: bass - POS: NOUN, noun, singular or mass - DEP: dobj
Token: bass - POS: NOUN, noun, singular or mass - DEP: dobj
Token: bass - POS: NOUN, noun, singular or mass - DEP: dobj
Token: Bass - POS: PROPN, noun, proper singular - DEP: compound


Bummer! All usages of the word 'bass' point to a 'thing,' which is represented in English by a noun (generic thing) or a proper noun (specific brand or shop).

Let us consider another example: the word 'beat'

In [5]:
inputText = [ 'Beat the cracked eggs on high for five minutes',
               'Zimbabwe beat Australia in a shock upset at the World Cup',
               'Let us move to the beat of the music',
               "Ah! the beat of a bird's wings, the smell of Spring flowers!",
               'I am completely beat!',
               "This is Inspector Lestrade's beat"
              ]

getPOSInformation('beat', inputText)

Token: Beat - POS: VERB, verb, base form - DEP: ROOT
Token: beat - POS: VERB, verb, past tense - DEP: ROOT
Token: beat - POS: NOUN, noun, singular or mass - DEP: pobj
Token: beat - POS: NOUN, noun, singular or mass - DEP: ROOT
Token: beat - POS: ADJ, adjective (English), other noun-modifier (Chinese) - DEP: acomp
Token: beat - POS: NOUN, noun, singular or mass - DEP: attr


As we can observe, the word 'beat' has multiple meanings, which translate into multiple POS tags. POS tagging can help in identifying which broader category of meaning is pointed at by the word 'beat' in a particular sentence.

Currently, WSD is an open problem, with many different approaches proposed as solutions.

## Dependency Parsing
Dependency parsing is a process of identifying the grammatical relationships between words in a sentence. `spaCy` provides a built-in dependency parser that allows us to examine the relationships between individual tokens, or phrases.

_Advantage of dependency parsing over POS tagging_: POS tagging usually relies on information from the most-immediate neaghbors of the current token. Dependency parsing enables tokens that may not be immediately adjacent to participate in the analysis process with the help of underlying grammar rules.

Applications of dependency parsing include chatbots, question-answering, and machine-translation.

Consider the following sentences:
- "I forwarded you the email"
- "You forwarded me the email"

If we parse and tokenize these sentences, and remove stop words, here is what we get:

#### Exercise (10 minutes)

Parse the following sentences and remove stop words:
- "I forwarded you the email"
- "You forwarded me the email"


In [21]:
# Write a solution to the exercise here
testInput = [
    "I forwarded you the email",
    "You forwarded me the email"
]

for sentence in testInput:
    doc = nlp(sentence)

    print(f"\nOriginal sentence: {sentence}")

    filtered_tokens = []

    for token in doc:
        if not token.is_stop:
            filtered_tokens.append(token.text)
    print("After stop word removal:", " ".join(filtered_tokens))


Original sentence: I forwarded you the email
After stop word removal: forwarded email

Original sentence: You forwarded me the email
After stop word removal: forwarded email


The above result is extremely undesirable since it equates two sentences with opposite meaning. The filtered tokens do not capture the subject of the action being performed, nor the object.

Let us check the sentences based on the `token.dep_` attribute.

In [7]:
getPOSInformation("*", testInput)

Token: I - POS: PRON, pronoun, personal - DEP: nsubj
Token: forwarded - POS: VERB, verb, past tense - DEP: ROOT
Token: you - POS: PRON, pronoun, personal - DEP: dative
Token: the - POS: DET, determiner - DEP: det
Token: email - POS: NOUN, noun, singular or mass - DEP: dobj
Token: You - POS: PRON, pronoun, personal - DEP: nsubj
Token: forwarded - POS: VERB, verb, past tense - DEP: ROOT
Token: me - POS: PRON, pronoun, personal - DEP: dative
Token: the - POS: DET, determiner - DEP: det
Token: email - POS: NOUN, noun, singular or mass - DEP: dobj


As we can see, the `dep_` tag carries the meaning of the token as it is intended in the sentence.

Here is a list of most common dependency tags used by spaCy:

- **amod**: Adjectival modifier. This word modifies the noun or pronoun in the sentence.
- **aux**: Auxiliary. This word defines the relationship between the token and the main verb of the sentence.
- **compound**: Compound. This word is composed of two or more tokens, usually nouns. The first word usually modifies the second word.
- **dative**: Dative object. This relationship signals an indirect association between the action verb and the token that receives the action.
- **det**: Determiner. This word identifies the noun phrase in the sentence.
- **dobj**: Direct object. This is a relation between the main verb and the object of the sentence.
- **nsubj**: Nominal subject. This relationship points to the subject of the sentence.
- **nsubjpass**: Nominal subject, passive. This relationship points to the passive subject of the sentence.
- **nummod**: Numeric modifier. This word modifies a noun with a number.
- **poss**: Possessive modifier. This word indicates the possessor of the noun phrase.
- **root**: The root. This is the main verb of the sentence.

The **ROOT** tag is a special tag used to identify the main verb of the sentence. If the input is not a complete sentence, but a clause or a phrase, then the **ROOT** tag is assigned to the root of the phrase, usually the head noun.

Each sentence or phrase has exactly one root element, which is the verb or the main predicate of the sentence. This element also serves as the top-most element of the parse tree.

Below we shall see some examples of dependency parsing in action on our previous sentences:


In [8]:
from spacy import displacy

s1 = inputText[0]
doc1 = nlp(s1)
displacy.render(doc1, style="dep", manual=False)

**Beat** is the main verb of the sentence, and as such, is the root of this tree.

The substring _"five minutes"_ is a noun phrase, but not a complete sentence (though it maybe used as such in conversation). Hence, the root of this subtree is the head noun (the main idea in the phrase) - _"minutes"_, and its numeric modifier (nummod) is the child node.

Similarly, the phrase _"the cracked eggs"_ is rooted on the head noun (_"eggs"_), with the determiner, and the adjective being assigned as the children.

In [9]:
# Display a dependency graph for sentence 4 - "I am completely beat!"

s1 = inputText[4]
doc1 = nlp(s1)
displacy.render(doc1, style="dep", manual=False)

In [10]:
# Display a dependency graph for sentence 5 - "This is Inspector Lestrade's beat"
s1 = inputText[5]
doc2 = nlp(s1)
displacy.render(doc2, style="dep", manual=False)

In [11]:
# Display a dependency graph for sentence 1 - "Zimbabwe beat Australia in a shock upset at the World Cup"
s1 = inputText[1]
doc3 = nlp(s1)
displacy.render(doc3, style="dep", manual=False)

## Named-Entity Recognition (NER)

What is a "named entity?" A named entity is a real-world object that we can refer to by a proper name or quantity of interest.

It can be a person, a place (city, country, landmark, or famous building), an organization, a company, a product, dates, times, percentages, monetary amounts, a drug, or a disease name. Some examples are Sherlock Holmes, Jane Austen, London, Reichenbach Falls, UN (United Nations), Google, Apple, Pound Sterling, Chilean Peso, KitKat, Nescafe, and so on.

A named entity always points to a specific object, and that object is distinguishable via the corresponding named entity. For instance, if we tag the sentence - _Paris is the capital of France_, we parse _Paris_ and _France_ as named entities, but not the word capital. The reason is that capital does not point to a specific object; it’s a general name for many objects.

In [23]:
# Let us see if spaCy recognizes stuff from our previous sentences
displacy.render(doc1, style="ent", manual=False)

In [13]:
# Display entities for the second sentence
displacy.render(doc2, style="ent", manual=False)

In [14]:
# Display entities for the third sentence
displacy.render(doc3, style="ent", manual=False)

Here is a list of the most common named entity types recognized by `spaCy`:
- PERSON: People, including fictional
- NORP: Nationalities or religious or political groups
- FAC: Buildings, airports, highways, bridges, and so on
- ORG: Companies, agencies, institutions, and so on
- GPE: Countries, cities, states
- LOC: Non-GPE locations, mountain ranges, bodies of water
- PRODUCT: Objects, vehicles, foods, and so on (not services)
- EVENT: Named hurricanes, battles, wars, sports events, and so on
- WORK_OF_ART: Titles of books, songs, and so on
- LAW: Named documents made into laws
- LANGUAGE: Any named language
- DATE: Absolute or relative dates or periods
- TIME: Times smaller than a day
- PERCENT: Percentage, including %
- MONEY: Monetary values, including unit
- QUANTITY: Measurements, as of weight or distance
- ORDINAL: first, second, and so on
- CARDINAL: Numerals that do not fall under another type


##  Merging and splitting tokens

Consider compound tokens such as `New York` or `St. Lucia`. These token groups should not be separated by the semantic analyzer, even if the raw whitespace tokenizer reports them as multiple tokens.

`spaCy` provides a very handy functionality to override the default tokenizer behavior through the method `doc.retokenize()`.

The retokenization is achieved via a `ContextManager` object, with the `span.merge()` method being used to merge a range of adjacent tokens into a single token. The method expects the range ot tokens, and their corresponding attributes as arguments.

Let's see some examples below:

In [15]:
doc = nlp("From tumpty gherkin to New York")
displacy.render(doc, style="ent")

"Tumpty Gherkin" (a fictional location) is not being recognized by `spaCy` as a compound token. We can add a rule to merge these two words **and also apply necessary attributes**.

In [16]:
print([(token.text, token.i, token.lemma_) for token in doc])

[('From', 0, 'from'), ('tumpty', 1, 'tumpty'), ('gherkin', 2, 'gherkin'), ('to', 3, 'to'), ('New', 4, 'New'), ('York', 5, 'York')]


In [17]:
with doc.retokenize() as retokenizer:
    retokenizer.merge(
        doc[1:3],  # Span (range of tokens) to merge
        attrs={"LEMMA": 'tumpty gherkin',  # Base representation of the merged token
               "ENT_TYPE": 'GPE',  # NER tag to be applied to the merged token
               "ENT_IOB": 'B'}  # Mark as the beginning of entity
    )


The `ENT_IOB` tag signifies the (B)egining, (I)nside, or (O)utside of a named-entity.

In [18]:
print([(token.text, token.i, token.lemma_) for token in doc])

[('From', 0, 'from'), ('tumpty gherkin', 1, 'tumpty gherkin'), ('to', 2, 'to'), ('New', 3, 'New'), ('York', 4, 'York')]


In [19]:
displacy.render(doc, style="ent")

Which is the intended output!

## Summary

In this notebook, we learned about the following topics:
- Part-of-Speech (POS) tagging
- Dependency parsing
- Named-Entity Recognition (NER)
- Merging and splitting tokens using `spaCy`'s `retokenize()` context manager